# Notebook 03b — Feature Selection (training data only) and the 2×2 Factorial Ablation

### What this notebook does
1. **Re-checks the imputation comparison (old Table 3)** on training respondents only, with corrected missing codes.
2. **Re-runs the five-method voting feature selection** — Chi-square, Mutual Information, RFE (Logistic Regression),
   Random Forest importance, LASSO — on a sample of the **training set only** (reviewer comment #1).
3. **Measures how stable the selection is** — across different K / vote thresholds, and across different random
   samples of respondents (replaces the old "93.3% overlap" claim).
4. **Runs the 2×2 factorial ablation** the reviewer asked for (comment #6):

|  | All features | Selected features |
|---|---|---|
| **Simple imputation** (median / most frequent) | cell 1 | cell 2 |
| **MissForest imputation** | cell 3 | cell 4 |

Every cell uses the **same split, the same models (Logistic Regression and LightGBM) and the same tuned
hyperparameters from Notebook 03**, so only one thing changes between neighbouring cells. Scores are on the
**validation set**; the test set is still untouched (it is used once, in Notebook 04).

**Input** — `brfss_model_ready_v2.csv`, `feature_config_v2.json`, `v2_best_params.json` (from Notebook 03)
**Output** — `outputs/v2_imputation_error.csv`, `v2_selection_votes.csv`, `v2_selection_sensitivity.csv`,
`v2_selection_stability.csv`, `v2_selected_features.json`, `v2_factorial_validation.csv`, `v2_factorial_effects.csv`,
`outputs/figures/fig_feature_votes.png`, `models/v2/factorial/*.joblib`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])

import pandas as pd
import numpy as np
import os, json, time, joblib, itertools
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
import sklearn, lightgbm

print(f"scikit-learn {sklearn.__version__} | lightgbm {lightgbm.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/stroke_prediction'
READY_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
CONFIG_PATH  = os.path.join(BASE_DIR, 'outputs', 'feature_config_v2.json')
OUTPUTS_DIR  = os.path.join(BASE_DIR, 'outputs')
FIG_DIR      = os.path.join(OUTPUTS_DIR, 'figures')
PARAMS_PATH  = os.path.join(OUTPUTS_DIR, 'v2_best_params.json')
FACT_DIR     = os.path.join(BASE_DIR, 'models', 'v2', 'factorial')
SRC_DIR      = os.path.join(BASE_DIR, 'src')
for d in [FIG_DIR, FACT_DIR, SRC_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Run settings ──────────────────────────────────────────────────────────
FS_BASE      = 'B'          # select from feature set B (no likely post-stroke features) — the main model
SAMPLE       = 'primary'
SEL_N        = 50_000       # training respondents used for each feature-selection run
K_REF        = 20           # each method nominates its top-K features ...
VOTE_REF     = 3            # ... and a feature is kept if at least this many of the 5 methods nominate it
K_GRID       = [10, 15, 20, 25, 30]
VOTE_GRID    = [2, 3, 4]
N_REPEATS    = 5            # extra selection runs on different random training samples (stability)
LASSO_C      = 0.01
MF_FIT_N     = 50_000       # MissForest is fitted on this many training respondents, then applied to everyone
MF_MAX_ITER  = 5
IMP_TEST_N   = 5_000        # sample size for the imputation-error comparison
N_BOOT       = 1000         # bootstrap resamples for confidence intervals of the factorial effects
SEED         = 42

config = json.load(open(CONFIG_PATH))
df = pd.read_csv(READY_PATH)
if SAMPLE == 'primary':
    df = df[df['meta_keep_primary']]
train = df[df['split'] == 'train'].reset_index(drop=True)
val   = df[df['split'] == 'validation'].reset_index(drop=True)
FEATURES = config[f'features_{FS_BASE}']
NUMBIN   = [f for f in config['continuous'] + config['ordinal'] + config['binary'] if f in FEATURES]
print(f"Feature set {FS_BASE}: {len(FEATURES)} features | train {len(train):,} | validation {len(val):,} | "
      f"test not loaded into any model")

### Helper file: the MissForest imputer

scikit-learn has no ready-made MissForest. The usual recipe — and the one the old notebooks used — is
`IterativeImputer` with a random-forest model: each column with blanks is predicted from all the other columns,
round after round. Two fixes compared with the old version:

* It is fitted on a sample of the **training data it is given** (never validation/test respondents).
* Yes/no and ordinal answers are **rounded** to a real answer, so there are no "0.37 smokers" (reviewer comment #7).

The class is saved as `src/stroke_utils.py` on your Drive, because saved models that contain it need to find it
again when Notebook 04 loads them.

In [ ]:
STROKE_UTILS = """
import warnings
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor


class MissForestImputer(BaseEstimator, TransformerMixin):
    \"\"\"MissForest-style imputation (IterativeImputer + random forests).
    Fitted on a stratified subsample of the data passed to fit (the training data);
    columns listed in round_cols (binary / ordinal) are rounded and clipped to their observed range.\"\"\"

    def __init__(self, n_fit=50000, max_iter=5, round_cols=(), random_state=42):
        self.n_fit = n_fit
        self.max_iter = max_iter
        self.round_cols = round_cols
        self.random_state = random_state

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        rng = np.random.default_rng(self.random_state)
        idx = np.arange(len(X))
        if len(X) > self.n_fit:
            if y is not None:
                y = np.asarray(y)
                idx = np.concatenate([
                    rng.choice(np.where(y == c)[0], int(round(self.n_fit * (y == c).mean())), replace=False)
                    for c in np.unique(y)])
            else:
                idx = rng.choice(idx, self.n_fit, replace=False)
        forest = RandomForestRegressor(n_estimators=50, max_depth=12, min_samples_leaf=5, max_features=0.5,
                                       n_jobs=-1, random_state=self.random_state)
        self.imputer_ = IterativeImputer(estimator=forest, max_iter=self.max_iter, initial_strategy='median',
                                         skip_complete=True, keep_empty_features=True,
                                         random_state=self.random_state).fit(X[idx])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            lo, hi = np.nanmin(X, axis=0), np.nanmax(X, axis=0)
        self.lo_ = np.where(np.isnan(lo), -np.inf, lo)
        self.hi_ = np.where(np.isnan(hi), np.inf, hi)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        Z = self.imputer_.transform(np.asarray(X, dtype=float))
        for j in self.round_cols:
            Z[:, j] = np.clip(np.round(Z[:, j]), self.lo_[j], self.hi_[j])
        return Z
"""
with open(os.path.join(SRC_DIR, 'stroke_utils.py'), 'w') as fh:
    fh.write(STROKE_UTILS)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import importlib, stroke_utils
importlib.reload(stroke_utils)
from stroke_utils import MissForestImputer
print(f"Saved {os.path.join(SRC_DIR, 'stroke_utils.py')}")

### Step 1 — Which imputation method reconstructs answers best? (redo of old Table 3)

Same idea as before: take people whose answers we **know**, hide 20% of those answers at random, let each
method guess them back, and measure the error (all columns scaled to 0–1 first, so every column counts equally).

Differences from the old Table 3:
* respondents come from the **training set** only, with the correct 4.2% stroke rate (old: 6.17%, the filtered sample);
* the features are the cleaned, codebook-checked feature set used for modelling;
* yes/no columns are included as well as numeric ones.

**Limitation to state in the chapter:** hiding answers *at random* is easier than real BRFSS missingness, which
follows skip logic and refusals. So this table shows how well each method fills random gaps — not proof that it is
best for this survey.

In [ ]:
def stratified_sample(d, n, seed):
    if n >= len(d):
        return d.copy()
    s, _ = train_test_split(d, train_size=n, stratify=d['stroke'], random_state=seed)
    return s

imp_sample = stratified_sample(train, IMP_TEST_N, SEED)
X_true = imp_sample[NUMBIN].to_numpy(float)
lo, hi = np.nanmin(X_true, axis=0), np.nanmax(X_true, axis=0)
X_true = (X_true - lo) / np.where(hi > lo, hi - lo, 1)          # scale every column to 0–1

rng = np.random.default_rng(SEED)
hide = (~np.isnan(X_true)) & (rng.random(X_true.shape) < 0.20)   # hide 20% of the KNOWN answers
X_hidden = X_true.copy()
X_hidden[hide] = np.nan

methods = {
    'Mean substitution'   : SimpleImputer(strategy='mean', keep_empty_features=True),
    'Median substitution' : SimpleImputer(strategy='median', keep_empty_features=True),
    'KNN (k = 5)'         : KNNImputer(n_neighbors=5, keep_empty_features=True),
    'MissForest'          : MissForestImputer(n_fit=IMP_TEST_N, max_iter=MF_MAX_ITER, random_state=SEED),
}
rows = []
for name, imputer in methods.items():
    start = time.time()
    X_filled = imputer.fit_transform(X_hidden)
    err = X_filled[hide] - X_true[hide]
    rows.append({'method': name, 'RMSE': np.sqrt(np.mean(err ** 2)), 'MAE': np.mean(np.abs(err)),
                 'seconds': time.time() - start})
imp_err = pd.DataFrame(rows).round(4)
imp_err.to_csv(os.path.join(OUTPUTS_DIR, 'v2_imputation_error.csv'), index=False)
print(f"{hide.sum():,} hidden answers across {len(NUMBIN)} numeric/ordinal/binary features, "
      f"{len(imp_sample):,} training respondents\n")
print(imp_err.to_string(index=False))

### Step 2 — Five-method voting feature selection (training data only)

Each method looks at a stratified sample of **training** respondents and scores every feature:

| Method | What it asks |
|---|---|
| Chi-square | Are the answers distributed differently in stroke vs non-stroke people? |
| Mutual Information | How much does knowing this answer reduce uncertainty about stroke? |
| RFE (Logistic Regression) | Which features survive when the weakest are removed one by one? |
| Random Forest importance | How much do the forest's splits on this feature improve purity? |
| LASSO (L1 Logistic Regression) | Which features keep a non-zero weight under a strong penalty? |

Each method nominates its **top K = 20**; a feature is kept if **at least 3 of the 5** nominate it. A nominal
feature (e.g. marital status) is one-hot encoded for scoring, and a feature's score is the best score of any of its
columns, including its "was missing" flag — so the vote is always about the **original question**.

In [ ]:
METHODS = ['Chi-square', 'Mutual information', 'RFE (LR)', 'Random forest', 'LASSO']

def design_matrix(d, features):
    """Numeric/binary features as one column (+ a 'was missing' flag if needed); nominal features one-hot.
    owner[j] = the original feature that column j belongs to."""
    cols, owner = [], []
    for f in features:
        x = d[f]
        if x.notna().sum() == 0:
            continue
        if f in config['nominal']:
            dummies = pd.get_dummies(x.fillna(-1).astype(int).astype(str), prefix=f, dtype=float)
            cols.append(dummies)
            owner += [f] * dummies.shape[1]
        else:
            fill = x.mode().iloc[0] if f in config['binary'] else x.median()
            cols.append(x.fillna(fill).rename(f))
            owner.append(f)
            if x.isna().any():
                cols.append(x.isna().astype(float).rename(f + '_missing'))
                owner.append(f)
    return pd.concat(cols, axis=1).to_numpy(float), np.array(owner)

def method_scores(d, features, seed):
    X, owner = design_matrix(d, features)
    y = d['stroke'].to_numpy()
    X_mm  = MinMaxScaler().fit_transform(X)
    X_std = StandardScaler().fit_transform(X)
    discrete = np.array([len(np.unique(X[:, j])) <= 10 for j in range(X.shape[1])])
    rfe = RFE(LogisticRegression(class_weight='balanced', max_iter=1000), n_features_to_select=1, step=1).fit(X_std, y)
    forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, max_features='sqrt',
                                    class_weight='balanced_subsample', n_jobs=-1, random_state=seed).fit(X, y)
    lasso = LogisticRegression(penalty='l1', solver='liblinear', C=LASSO_C, class_weight='balanced',
                               random_state=seed).fit(X_std, y)
    col_scores = {
        'Chi-square'        : chi2(X_mm, y)[0],
        'Mutual information': mutual_info_classif(X_mm, y, discrete_features=discrete, random_state=seed),
        'RFE (LR)'          : (X.shape[1] + 1 - rfe.ranking_).astype(float),   # higher = survived longer
        'Random forest'     : forest.feature_importances_,
        'LASSO'             : np.abs(lasso.coef_[0]),
    }
    per_feature = {m: pd.Series(np.nan_to_num(v), index=owner).groupby(level=0).max() for m, v in col_scores.items()}
    return pd.DataFrame(per_feature).reindex(features).fillna(0)[METHODS]

def vote(scores, k, threshold):
    """Each method nominates its top-k features (score > 0); keep features with >= threshold nominations."""
    top = {m: set(scores[m][scores[m] > 0].nlargest(k).index) for m in METHODS}
    votes = pd.Series({f: sum(f in top[m] for m in METHODS) for f in scores.index})
    return votes, top, set(votes[votes >= threshold].index)

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

start = time.time()
scores_ref = method_scores(stratified_sample(train, SEL_N, SEED), FEATURES, SEED)
votes_ref, top_ref, selected_set = vote(scores_ref, K_REF, VOTE_REF)
print(f"Reference selection run on {min(SEL_N, len(train)):,} training respondents: {(time.time()-start)/60:.1f} min")

In [ ]:
vote_table = pd.DataFrame({
    **{f'top{K_REF}_{m}': [int(f in top_ref[m]) for f in FEATURES] for m in METHODS},
    **{f'rank_{m}': scores_ref[m].rank(ascending=False, method='min').astype(int).values for m in METHODS},
    'votes': votes_ref.reindex(FEATURES).values,
}, index=FEATURES)
vote_table['mean_rank'] = vote_table[[f'rank_{m}' for m in METHODS]].mean(axis=1).round(1)
vote_table['selected'] = vote_table.index.isin(selected_set)
vote_table = vote_table.sort_values(['votes', 'mean_rank'], ascending=[False, True])
vote_table.to_csv(os.path.join(OUTPUTS_DIR, 'v2_selection_votes.csv'), index_label='feature')

SELECTED = [f for f in vote_table.index if f in selected_set]     # ordered by votes, then mean rank
json.dump({'base_feature_set': FS_BASE, 'k': K_REF, 'vote_threshold': VOTE_REF, 'selection_n': min(SEL_N, len(train)),
           'n_candidates': len(FEATURES), 'features_S': SELECTED},
          open(os.path.join(OUTPUTS_DIR, 'v2_selected_features.json'), 'w'), indent=2)

print(f"Selected {len(SELECTED)} of {len(FEATURES)} features (top {K_REF} per method, >= {VOTE_REF} of 5 votes)\n")
print(vote_table[['votes', 'mean_rank', 'selected']].to_string())

# Figure: which method nominated which feature (300 DPI for the publisher)
grid = vote_table[[f'top{K_REF}_{m}' for m in METHODS]].to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 0.22 * len(vote_table) + 1.5))
ax.imshow(grid, cmap='Blues', aspect='auto', vmin=0, vmax=1.4)
ax.set_xticks(range(len(METHODS)), METHODS, rotation=30, ha='right', fontsize=8)
ax.set_yticks(range(len(vote_table)), [f"{f}  ({v})" for f, v in zip(vote_table.index, vote_table['votes'])], fontsize=7)
n_sel = len(SELECTED)
if 0 < n_sel < len(vote_table):
    ax.axhline(n_sel - 0.5, color='black', linewidth=1)
ax.set_title(f'Feature nominations (top {K_REF} per method); features above the line have ≥ {VOTE_REF} votes', fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_feature_votes.png'), dpi=300)
plt.show()

### Step 3 — Is the selection stable?

**(a) Settings.** Re-vote with K = 10, 15, 20, 25, 30 and thresholds 2, 3, 4 (15 combinations — the same grid as
the old "93.3%" check). For each, the **Jaccard similarity** with the reference selection: shared features ÷ all
features in either set (1 = identical, 0 = nothing in common). The old 93.3% was most likely "14 of 15
combinations gave the same set"; Jaccard is the standard measure and is what the chapter should report.

**(b) People.** Repeat the whole selection on **5 different random samples** of training respondents. A feature
picked in 6 of 6 runs is a robust choice; one picked in 2 of 6 is a coin-flip.

In [ ]:
sens_rows = []
for k, t in itertools.product(K_GRID, VOTE_GRID):
    _, _, s = vote(scores_ref, k, t)
    sens_rows.append({'k': k, 'vote_threshold': t, 'n_selected': len(s), 'jaccard_vs_reference': round(jaccard(s, selected_set), 3),
                      'identical': s == selected_set})
sens = pd.DataFrame(sens_rows)
sens.to_csv(os.path.join(OUTPUTS_DIR, 'v2_selection_sensitivity.csv'), index=False)
print("(a) Sensitivity to K and vote threshold")
print(sens.pivot(index='k', columns='vote_threshold', values='jaccard_vs_reference').to_string())
print(f"Mean Jaccard {sens['jaccard_vs_reference'].mean():.3f} | identical to reference in "
      f"{sens['identical'].sum()} of {len(sens)} combinations")

runs = [selected_set]
for r in range(1, N_REPEATS + 1):
    start = time.time()
    _, _, s = vote(method_scores(stratified_sample(train, SEL_N, SEED + r), FEATURES, SEED + r), K_REF, VOTE_REF)
    runs.append(s)
    print(f"  resample {r}: {len(s)} selected, Jaccard vs reference {jaccard(s, selected_set):.3f} "
          f"({(time.time()-start)/60:.1f} min)")

pairwise = [jaccard(a, b) for a, b in itertools.combinations(runs, 2)]
freq = pd.Series({f: sum(f in s for s in runs) for f in FEATURES}).sort_values(ascending=False)
stab = pd.DataFrame({'times_selected': freq, 'out_of': len(runs), 'in_reference': freq.index.isin(selected_set)})
stab.to_csv(os.path.join(OUTPUTS_DIR, 'v2_selection_stability.csv'), index_label='feature')
print(f"\n(b) Across {len(runs)} runs on different training samples: mean pairwise Jaccard "
      f"{np.mean(pairwise):.3f} (min {np.min(pairwise):.3f})")
print(f"Features chosen in every run: {int((freq == len(runs)).sum())} | "
      f"chosen in some runs only: {int(((freq > 0) & (freq < len(runs))).sum())}")
print(stab[stab['times_selected'] > 0].to_string())

### Step 4 — The 2×2 factorial ablation (reviewer comment #6)

Four preprocessing set-ups × two models (Logistic Regression, LightGBM) = 8 models, all trained on the full
training set and scored on the validation set.

* **Imputation** is the only difference between rows: simple (training median / most frequent) vs MissForest.
  Both rows also get the same "was missing" flags and the same "missing" category for nominal features, so the
  only thing that changes is the *value* written into the blank.
* **Feature set** is the only difference between columns: all features vs the selected ones.
* **Hyperparameters are fixed** at the Notebook 03 values, so tuning luck cannot explain a difference.

Each finished model is saved; re-running the notebook loads it instead of retraining.

In [ ]:
best_params = json.load(open(PARAMS_PATH))
LR_PARAMS   = best_params[f'{SAMPLE}_{FS_BASE}_LogisticRegression']
LGBM_PARAMS = best_params[f'{SAMPLE}_{FS_BASE}_LightGBM']
MODELS = ['LogisticRegression', 'LightGBM']
print("Fixed hyperparameters from Notebook 03:")
print("  LogisticRegression", LR_PARAMS)
print("  LightGBM          ", LGBM_PARAMS)

def make_model(name):
    if name == 'LogisticRegression':
        return LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=2000, random_state=SEED, **LR_PARAMS)
    return LGBMClassifier(class_weight='balanced', subsample_freq=1, n_jobs=-1, random_state=SEED, verbose=-1, **LGBM_PARAMS)

def make_prep(features, imputation):
    num  = [f for f in config['continuous'] + config['ordinal'] if f in features]
    binr = [f for f in config['binary'] if f in features]
    nom  = [f for f in config['nominal'] if f in features]
    if imputation == 'simple':
        fill = ColumnTransformer([('num', SimpleImputer(strategy='median'), num),
                                  ('bin', SimpleImputer(strategy='most_frequent'), binr)])
    else:
        to_round = [i for i, f in enumerate(num + binr) if f in config['ordinal'] or f in config['binary']]
        fill = MissForestImputer(n_fit=MF_FIT_N, max_iter=MF_MAX_ITER, round_cols=to_round, random_state=SEED)
    return ColumnTransformer([
        ('fill',  fill, num + binr),
        ('flags', MissingIndicator(features='missing-only', error_on_new=False), num + binr),
        ('nom',   Pipeline([('impute', SimpleImputer(strategy='constant', fill_value=-1)),
                            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), nom),
    ])

FEATURE_VARIANTS = {'all': FEATURES, 'selected': SELECTED}
IMPUTATIONS = ['simple', 'missforest']
y_train, y_val = train['stroke'].to_numpy(), val['stroke'].to_numpy()
val_pred, fact_rows = {}, []

for imp, (fv, feats) in itertools.product(IMPUTATIONS, FEATURE_VARIANTS.items()):
    paths = {m: os.path.join(FACT_DIR, f'{SAMPLE}_{FS_BASE}_{imp}_{fv}_{m}.joblib') for m in MODELS}
    if all(os.path.exists(p) for p in paths.values()):
        pipes = {m: joblib.load(p) for m, p in paths.items()}
        print(f"{imp:<10} | {fv:<8} | loaded from disk")
    else:
        start = time.time()
        prep = make_prep(feats, imp)
        X_tr = prep.fit_transform(train[feats], y_train)
        prep_min = (time.time() - start) / 60
        scaler = StandardScaler().fit(X_tr)
        pipes = {
            'LogisticRegression': Pipeline([('prep', prep), ('scale', scaler),
                                            ('model', make_model('LogisticRegression').fit(scaler.transform(X_tr), y_train))]),
            'LightGBM': Pipeline([('prep', prep), ('model', make_model('LightGBM').fit(X_tr, y_train))]),
        }
        for m in MODELS:
            joblib.dump(pipes[m], paths[m])
        print(f"{imp:<10} | {fv:<8} | {len(feats)} features -> {X_tr.shape[1]} columns | "
              f"imputation {prep_min:.1f} min | total {(time.time()-start)/60:.1f} min")
    for m in MODELS:
        p = pipes[m].predict_proba(val[feats])[:, 1]
        val_pred[(m, imp, fv)] = p
        fact_rows.append({'model': m, 'imputation': imp, 'features': fv, 'n_features': len(feats),
                          'val_auc': round(roc_auc_score(y_val, p), 4),
                          'val_ap': round(average_precision_score(y_val, p), 4)})

fact = pd.DataFrame(fact_rows)
fact.to_csv(os.path.join(OUTPUTS_DIR, 'v2_factorial_validation.csv'), index=False)
for m in MODELS:
    print(f"\n{m} — validation AUC (AP in brackets)")
    t = fact[fact['model'] == m].set_index(['imputation', 'features'])
    print(pd.DataFrame({fv: [f"{t.loc[(imp, fv), 'val_auc']:.4f} ({t.loc[(imp, fv), 'val_ap']:.4f})" for imp in IMPUTATIONS]
                        for fv in FEATURE_VARIANTS}, index=IMPUTATIONS).to_string())

### Step 5 — How big is each effect, and could it be chance?

From the four cells we compute three numbers per model:

* **Imputation effect** = MissForest − simple, averaged over the two feature sets
* **Selection effect** = selected − all features, averaged over the two imputation methods
* **Interaction** = does selection help more (or less) under MissForest than under simple imputation?

The 95% confidence intervals come from **bootstrapping the validation set**: draw 64,000 respondents with
replacement 1,000 times, recompute all four AUCs on each draw, and take the middle 95% of each effect.
**If an interval contains 0, the data cannot tell that effect apart from no effect.**

In [ ]:
def effects(a):
    """a: dict (imputation, features) -> AUC. Returns the three factorial effects."""
    return {'Imputation (MissForest - simple)': ((a[('missforest', 'all')] - a[('simple', 'all')]) +
                                                 (a[('missforest', 'selected')] - a[('simple', 'selected')])) / 2,
            'Feature selection (selected - all)': ((a[('simple', 'selected')] - a[('simple', 'all')]) +
                                                   (a[('missforest', 'selected')] - a[('missforest', 'all')])) / 2,
            'Interaction': ((a[('missforest', 'selected')] - a[('missforest', 'all')]) -
                            (a[('simple', 'selected')] - a[('simple', 'all')]))}

cells = list(itertools.product(IMPUTATIONS, FEATURE_VARIANTS))
eff_rows = []
for m in MODELS:
    point = effects({c: roc_auc_score(y_val, val_pred[(m, *c)]) for c in cells})
    rng = np.random.default_rng(SEED)
    boot = {e: [] for e in point}
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(y_val), len(y_val))
        if y_val[idx].min() == y_val[idx].max():
            continue
        for e, v in effects({c: roc_auc_score(y_val[idx], val_pred[(m, *c)][idx]) for c in cells}).items():
            boot[e].append(v)
    for e, v in point.items():
        lo_ci, hi_ci = np.percentile(boot[e], [2.5, 97.5])
        eff_rows.append({'model': m, 'effect': e, 'auc_change': round(v, 4), 'ci_low': round(lo_ci, 4),
                         'ci_high': round(hi_ci, 4), 'ci_includes_0': bool(lo_ci <= 0 <= hi_ci)})

eff = pd.DataFrame(eff_rows)
eff.to_csv(os.path.join(OUTPUTS_DIR, 'v2_factorial_effects.csv'), index=False)
print(eff.to_string(index=False))

print("\n" + "=" * 60)
print("  Notebook 03b complete")
print("=" * 60)
print(f"  Selected features : {len(SELECTED)} of {len(FEATURES)} -> outputs/v2_selected_features.json")
print(f"  Factorial models  : {FACT_DIR}")
print(f"  Tables            : outputs/v2_imputation_error.csv, v2_selection_*.csv, v2_factorial_*.csv")
print(f"  Figure (300 DPI)  : {os.path.join(FIG_DIR, 'fig_feature_votes.png')}")
print("=" * 60)